In [1]:
!pip install pandas

In [7]:
import requests
import pandas as pd
import time

# Semantic Scholar API base URL
BASE_URL = "https://api.semanticscholar.org/graph/v1/paper/"

# We specifically ask for the title, year, abstract, and the exact citation sentences (contexts)
FIELDS = "title,year,abstract,citations.paperId,citations.title,citations.year,citations.contexts"

# def get_citations(paper_id):
#     """Fetches the paper and all papers that cite it."""
#     url = f"{BASE_URL}{paper_id}?fields={FIELDS}&limit=100"
    
#     # Be polite to the API to avoid rate limits
#     time.sleep(4) 
    
#     try:
#         response = requests.get(url)
#         response.raise_for_status()
#         return response.json()
#     except requests.exceptions.RequestException as e:
#         print(f"Error fetching data for {paper_id}: {e}")
#         return None

def get_citations(paper_id, max_retries=3):
    """Fetches data with browser disguise and automatic retry logic for 429 errors."""
    url = f"{BASE_URL}{paper_id}?fields={FIELDS}&limit=100"
    
    # Disguise the Python script as a standard Chrome web browser
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json"
    }
    
    for attempt in range(max_retries):
        time.sleep(3) # Base polite delay
        
        try:
            response = requests.get(url, headers=headers)
            
            # If successful, return the data immediately
            if response.status_code == 200:
                return response.json()
                
            # If rate limited, wait longer and try again (Exponential Backoff)
            elif response.status_code == 429:
                wait_time = 10 * (attempt + 1)
                print(f"API Rate Limit hit (429). Waiting {wait_time} seconds before retrying...")
                time.sleep(wait_time)
                
            else:
                print(f"Failed with Status Code {response.status_code}: {response.text}")
                return None
                
        except requests.exceptions.RequestException as e:
            print(f"Network error: {e}")
            time.sleep(5)
            
    print(f"Failed to fetch {paper_id} after {max_retries} attempts.")
    return None

def build_snowball_dataset(seed_id, max_papers=5000):
    edges = []
    nodes = {} # Use a dictionary to avoid duplicate papers
    
    print(f"Starting Snowball Sampling from Seed ID: {seed_id}")
    seed_data = get_citations(seed_id)
    
    if not seed_data or 'citations' not in seed_data:
        print("Failed to retrieve seed data.")
        return None, None

    # Save the seed node
    nodes[seed_data['paperId']] = {
        'paperId': seed_data['paperId'],
        'title': seed_data.get('title'),
        'year': seed_data.get('year'),
        'abstract': seed_data.get('abstract')
    }

    queue = [seed_data['paperId']]
    visited = set()
    
    # Hop 1 & 2 Loop
    while queue and len(nodes) < max_papers:
        current_paper = queue.pop(0)
        
        if current_paper in visited:
            continue
            
        visited.add(current_paper)
        print(f"Processing paper {current_paper} | Total Unique Papers so far: {len(nodes)}")
        
        data = get_citations(current_paper)
        if not data or 'citations' not in data:
            continue
            
        for citation in data['citations']:
            cited_by_id = citation.get('paperId')
            
            # 1. Save the Node Data (if we don't have it yet)
            if cited_by_id and cited_by_id not in nodes:
                nodes[cited_by_id] = {
                    'paperId': cited_by_id,
                    'title': citation.get('title'),
                    'year': citation.get('year'),
                    'abstract': None # Abstract isn't in the citation payload, we'd need a separate call later
                }
                queue.append(cited_by_id) # Add to queue for the next hop
            
            # 2. Save the Edge Data (The connection + the citation sentence)
            contexts = citation.get('contexts')
            citation_sentence = contexts[0] if contexts else "No context provided"
            
            edges.append({
                'cited_paper': current_paper,
                'citing_paper': cited_by_id,
                'citation_sentence': citation_sentence
            })
            
            if len(nodes) >= max_papers:
                break

    print("Snowball complete!")
    return pd.DataFrame(nodes.values()), pd.DataFrame(edges)

# --- EXECUTION ---
# Example: "Attention Is All You Need" (A highly cited paper to test the script quickly)
# Real ID format looks like: 649def34f8be52c8b66281af98ae884c09aef38b
# TEST_SEED_ID = "204e3073870fae3d05bcbc2f6a8e263d9b72e776" 

# nodes_df, edges_df = build_snowball_dataset(TEST_SEED_ID, max_papers=500) # Testing with 500 first

# # Save to Parquet for our Apache Spark architecture!
# if nodes_df is not None:
#     nodes_df.to_parquet('nodes.parquet', index=False)
#     edges_df.to_parquet('edges.parquet', index=False)
#     print("Saved nodes.parquet and edges.parquet successfully.")
TEST_SEED_ID = "CorpusID:252514561" 

nodes_df, edges_df = build_snowball_dataset(TEST_SEED_ID, max_papers=50) # Just test 50 to see it work

if nodes_df is not None:
    nodes_df.to_parquet('nodes.parquet', index=False)
    edges_df.to_parquet('edges.parquet', index=False)
    print("Saved successfully!")

Starting Snowball Sampling from Seed ID: CorpusID:252514561
API Rate Limit hit (429). Waiting 10 seconds before retrying...
API Rate Limit hit (429). Waiting 20 seconds before retrying...
API Rate Limit hit (429). Waiting 30 seconds before retrying...
Failed to fetch CorpusID:252514561 after 3 attempts.
Failed to retrieve seed data.


In [14]:
import urllib.request
import tarfile
import pandas as pd
import os

print("1. Downloading SciCite directly from AllenAI...")
url = "https://s3-us-west-2.amazonaws.com/ai2-s2-research/scicite/scicite.tar.gz"
file_name = "scicite.tar.gz"
extract_dir = "scicite_raw_data"

# Download the file (if you haven't already, you can comment this out if it's already downloaded)
if not os.path.exists(file_name):
    urllib.request.urlretrieve(url, file_name)

print("2. Extracting files...")
with tarfile.open(file_name, "r:gz") as tar:
    if hasattr(tarfile, 'data_filter'):
        tar.extractall(path=extract_dir, filter='data')
    else:
        tar.extractall(path=extract_dir)

print("3. Finding and Loading JSONL into Pandas DataFrame...")
# Dynamically search for the file to avoid nested folder errors
train_file_path = None
for root, dirs, files in os.walk(extract_dir):
    if "train.jsonl" in files:
        train_file_path = os.path.join(root, "train.jsonl")
        break

if train_file_path is None:
    print("Error: Could not find train.jsonl in the extracted files!")
else:
    print(f"Found the file at: {train_file_path}")
    
    # Read the JSONL file
    df_train = pd.read_json(train_file_path, lines=True)

    print("\n4. Previewing the Data:")
    print(df_train[['string', 'label', 'citingPaperId', 'citedPaperId']].head())

    print("\n5. Saving to Parquet for Apache Spark...")
    # df_train.to_parquet("scicite_training_data.parquet", index=False)
    # Tell Pandas to use fastparquet instead of pyarrow
    df_train.to_parquet("scicite_training_data.parquet", index=False, engine="fastparquet")
    print("Success! Data is ready for the HGNN.")

1. Downloading SciCite directly from AllenAI...
2. Extracting files...
3. Finding and Loading JSONL into Pandas DataFrame...
Found the file at: scicite_raw_data/scicite/train.jsonl

4. Previewing the Data:
                                              string       label  \
0  However, how frataxin interacts with the Fe-S ...  background   
1  In the study by Hickey et al. (2012), spikes w...  background   
2  The drug also reduces catecholamine secretion,...  background   
3  By clustering with lowly aggressive close kin ...  background   
4  Ophthalmic symptoms are rare manifestations of...  background   

                              citingPaperId  \
0  1872080baa7d30ec8fb87be9a65358cd3a7fb649   
1  ce1d09a4a3a8d7fd3405b9328f65f00c952cf64b   
2  9cdf605beb1aa1078f235c4332b3024daa8b31dc   
3  d9f3207db0c79a3b154f3875c9760cc6b056904b   
4  88b86556857f4374842d2af2e359576806239175   

                               citedPaperId  
0  894be9b4ea46a5c422e81ef3c241072d4c73fdc0  
1  b6642e1

In [15]:
!pip install torch torch_geometric

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 MB 1.8 MB/s  0:00:41m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 1.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 1.5 MB/s  0:00:00 eta 0:00:01
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached markupsafe-3.0.3-cp313-cp313-macosx_11_0_arm64.whl (12 kB)
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
  

In [18]:
import pandas as pd
import torch
from torch_geometric.data import Data

print("1. Loading the Parquet file...")
df = pd.read_parquet("scicite_training_data.parquet", engine="fastparquet")

print("2. Creating the Node ID Mapping...")
# Extract all unique paper IDs from both columns
unique_papers = pd.concat([df['citingPaperId'], df['citedPaperId']]).unique()

# Create a mapping from the string hash to a PyTorch-friendly integer (0 to N-1)
paper_to_idx = {paper_id: idx for idx, paper_id in enumerate(unique_papers)}
num_nodes = len(unique_papers)
print(f"Total Unique Papers (Nodes) in the graph: {num_nodes}")

print("3. Building the Edge Index Tensor...")
# Map the string IDs in our dataframe to the new integers
source_nodes = df['citingPaperId'].map(paper_to_idx).values
target_nodes = df['citedPaperId'].map(paper_to_idx).values

# PyTorch Geometric expects edges in a [2, num_edges] format
# edge_index = torch.tensor([source_nodes, target_nodes], dtype=torch.long)
import numpy as np
edge_array = np.array([source_nodes, target_nodes])
edge_index = torch.tensor(edge_array, dtype=torch.long)
print(f"Edge Index Shape: {edge_index.shape}")

print("4. Encoding the Edge Labels (The Citation Intents)...")
# Map the text labels to numerical classes for classification
label_mapping = {'background': 0, 'method': 1, 'result': 2}
df['label_idx'] = df['label'].map(label_mapping)

edge_labels = torch.tensor(df['label_idx'].values, dtype=torch.long)

print("5. Constructing the PyTorch Geometric Data Object...")
# Create the foundational graph object!
graph_data = Data(edge_index=edge_index, y=edge_labels, num_nodes=num_nodes)

print("\n--- Final Graph Structure ---")
print(graph_data)

1. Loading the Parquet file...
2. Creating the Node ID Mapping...
Total Unique Papers (Nodes) in the graph: 9221
3. Building the Edge Index Tensor...
Edge Index Shape: torch.Size([2, 8243])
4. Encoding the Edge Labels (The Citation Intents)...
5. Constructing the PyTorch Geometric Data Object...

--- Final Graph Structure ---
Data(edge_index=[2, 8243], y=[8243], num_nodes=9221)


In [19]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from torch_geometric.data import Data

print("1. Loading Data and SciBERT...")
df = pd.read_parquet("scicite_training_data.parquet", engine="fastparquet")

# Load the tokenizer and model from Hugging Face
tokenizer = AutoTokenizer.from_pretrained('allenai/scibert_scivocab_uncased')
model = AutoModel.from_pretrained('allenai/scibert_scivocab_uncased')

# Push model to GPU if you have an Apple Silicon Mac (mps) or NVIDIA (cuda)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
model = model.to(device)
model.eval() # Set to evaluation mode (we are extracting features, not training SciBERT)

print(f"Using device: {device}")
print("2. Generating SciBERT Edge Embeddings (This will take a few minutes)...")

# We will process the text in small batches to save RAM
batch_size = 32
all_embeddings = []
sentences = df['string'].tolist()

with torch.no_grad(): # Don't track gradients, saves massive amounts of memory
    for i in range(0, len(sentences), batch_size):
        batch_texts = sentences[i:i+batch_size]
        
        # Tokenize the text
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        
        # Pass through SciBERT
        outputs = model(**inputs)
        
        # We take the [CLS] token embedding (the 0th index) as the representation of the whole sentence
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        
        # Move back to CPU and save
        all_embeddings.append(cls_embeddings.cpu())
        
        if i % 320 == 0:
            print(f"Processed {i}/{len(sentences)} citations...")

# Combine all batches into one massive tensor
edge_attr = torch.cat(all_embeddings, dim=0)
print(f"Edge Attributes Shape: {edge_attr.shape} (Should be [8243, 768])")

print("\n3. Rebuilding the PyTorch Geometric Graph...")
# (Using your previous code with the NumPy fix)
unique_papers = pd.concat([df['citingPaperId'], df['citedPaperId']]).unique()
paper_to_idx = {paper_id: idx for idx, paper_id in enumerate(unique_papers)}
num_nodes = len(unique_papers)

source_nodes = df['citingPaperId'].map(paper_to_idx).values
target_nodes = df['citedPaperId'].map(paper_to_idx).values
edge_index = torch.tensor(np.array([source_nodes, target_nodes]), dtype=torch.long)

label_mapping = {'background': 0, 'method': 1, 'result': 2}
edge_labels = torch.tensor(df['label'].map(label_mapping).values, dtype=torch.long)

# Because we don't have the full paper abstracts yet, we will initialize dummy node features 
# just to keep the PyTorch math happy. We will upgrade this later.
x = torch.ones((num_nodes, 1), dtype=torch.float)

# The Final Master Graph Object
graph_data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=edge_labels)

print("\n--- The Complete Semantic Graph ---")
print(graph_data)

# Save this graph object so you don't have to run SciBERT every time!
torch.save(graph_data, 'scicite_graph.pt')
print("Graph saved successfully as 'scicite_graph.pt'")

1. Loading Data and SciBERT...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 66709.28it/s]
BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using device: mps
2. Generating SciBERT Edge Embeddings (This will take a few minutes)...
Processed 0/8243 citations...
Processed 320/8243 citations...
Processed 640/8243 citations...
Processed 960/8243 citations...
Processed 1280/8243 citations...
Processed 1600/8243 citations...
Processed 1920/8243 citations...
Processed 2240/8243 citations...
Processed 2560/8243 citations...
Processed 2880/8243 citations...
Processed 3200/8243 citations...
Processed 3520/8243 citations...
Processed 3840/8243 citations...
Processed 4160/8243 citations...
Processed 4480/8243 citations...
Processed 4800/8243 citations...
Processed 5120/8243 citations...
Processed 5440/8243 citations...
Processed 5760/8243 citations...
Processed 6080/8243 citations...
Processed 6400/8243 citations...
Processed 6720/8243 citations...
Processed 7040/8243 citations...
Processed 7360/8243 citations...
Processed 7680/8243 citations...
Processed 8000/8243 citations...
Edge Attributes Shape: torch.Size([8243, 768]) (Should be 

In [21]:
import torch
import torch.nn.functional as F
import numpy as np
from model import SemanticEdgeClassifier # Assuming you saved the model class in model.py

print("1. Loading Graph and Initializing Model...")
# Load the graph we generated in Phase 1
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
graph_data = torch.load('scicite_graph.pt', map_location=device, weights_only=False)

# Instantiate the model and move to M4 GPU
model = SemanticEdgeClassifier(
    node_in_dim=1, 
    edge_in_dim=768, 
    hidden_dim=64, 
    num_classes=3
).to(device)

print("2. Generating Edge Splits (Train/Val/Test)...")
num_edges = graph_data.num_edges

# Create random permutation of edge indices
indices = np.random.permutation(num_edges)

# Split: 80% Train, 10% Validation, 10% Test
train_idx = indices[:int(0.8 * num_edges)]
val_idx = indices[int(0.8 * num_edges):int(0.9 * num_edges)]
test_idx = indices[int(0.9 * num_edges):]

# Create boolean masks for PyTorch Geometric
graph_data.train_mask = torch.zeros(num_edges, dtype=torch.bool)
graph_data.val_mask = torch.zeros(num_edges, dtype=torch.bool)
graph_data.test_mask = torch.zeros(num_edges, dtype=torch.bool)

graph_data.train_mask[train_idx] = True
graph_data.val_mask[val_idx] = True
graph_data.test_mask[test_idx] = True

print(f"Edges -> Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

print("3. Setting up Optimizer and Loss Function...")
# Adam optimizer is standard for GNNs
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.CrossEntropyLoss()

def train():
    model.train()
    optimizer.zero_grad()
    
    # Forward pass
    out = model(graph_data.x, graph_data.edge_index, graph_data.edge_attr)
    
    # Calculate loss ONLY on the training edges
    loss = criterion(out[graph_data.train_mask], graph_data.y[graph_data.train_mask])
    
    # Backpropagation
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad() # Disable gradient tracking for evaluation to save memory
def test(mask):
    model.eval()
    out = model(graph_data.x, graph_data.edge_index, graph_data.edge_attr)
    
    # Get the highest probability class
    pred = out.argmax(dim=1)  
    
    # Compare predictions to ground truth labels
    correct = pred[mask] == graph_data.y[mask]
    acc = int(correct.sum()) / int(mask.sum())
    return acc

print("\n4. Starting Training Loop...")
epochs = 100

for epoch in range(1, epochs + 1):
    loss = train()
    
    # Print metrics every 10 epochs
    if epoch % 10 == 0:
        train_acc = test(graph_data.train_mask)
        val_acc = test(graph_data.val_mask)
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}')

print("\n5. Final Evaluation...")
test_acc = test(graph_data.test_mask)
print(f'Final Test Accuracy (Unseen Data): {test_acc:.4f}')

# Save the trained weights!
torch.save(model.state_dict(), 'semantic_edge_classifier.pth')
print("Model weights saved successfully.")

1. Loading Graph and Initializing Model...
2. Generating Edge Splits (Train/Val/Test)...
Edges -> Train: 6594 | Val: 824 | Test: 825
3. Setting up Optimizer and Loss Function...

4. Starting Training Loop...
Epoch: 010, Loss: 0.5937, Train Acc: 0.7361, Val Acc: 0.7512
Epoch: 020, Loss: 0.5157, Train Acc: 0.8192, Val Acc: 0.8058
Epoch: 030, Loss: 0.4873, Train Acc: 0.8276, Val Acc: 0.8180
Epoch: 040, Loss: 0.4547, Train Acc: 0.8361, Val Acc: 0.8216
Epoch: 050, Loss: 0.4371, Train Acc: 0.8471, Val Acc: 0.8204
Epoch: 060, Loss: 0.4238, Train Acc: 0.8505, Val Acc: 0.8289
Epoch: 070, Loss: 0.4001, Train Acc: 0.8591, Val Acc: 0.8313
Epoch: 080, Loss: 0.3836, Train Acc: 0.8665, Val Acc: 0.8313
Epoch: 090, Loss: 0.3731, Train Acc: 0.8726, Val Acc: 0.8350
Epoch: 100, Loss: 0.3580, Train Acc: 0.8794, Val Acc: 0.8337

5. Final Evaluation...
Final Test Accuracy (Unseen Data): 0.8388
Model weights saved successfully.


In [22]:
import torch
import pandas as pd
import numpy as np
from model import SemanticEdgeClassifier 

print("1. Loading Model and Graph...")
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
graph_data = torch.load('scicite_graph.pt', map_location=device, weights_only=False)

model = SemanticEdgeClassifier(node_in_dim=1, edge_in_dim=768, hidden_dim=64, num_classes=3).to(device)
model.load_state_dict(torch.load('semantic_edge_classifier.pth', map_location=device, weights_only=True))
model.eval()

print("2. Predicting All Edge Intents...")
with torch.no_grad():
    # Pass the entire graph through the network
    out = model(graph_data.x, graph_data.edge_index, graph_data.edge_attr)
    predictions = out.argmax(dim=1)  # Get the predicted class (0, 1, or 2)

print("3. Mapping Intents to Semantic Weights...")
# Initialize a tensor of 1.0 weights
edge_weights = torch.ones_like(predictions, dtype=torch.float)

# Apply our semantic mathematical mapping
edge_weights[predictions == 1] = 1.5  # Method -> 1.5x credit
edge_weights[predictions == 2] = 1.2  # Result -> 1.2x credit

print("4. Calculating Semantic hp-frac Scores...")
# The target nodes are the bottom row of our edge_index [2, num_edges]
target_nodes = graph_data.edge_index[1]
num_nodes = graph_data.num_nodes

# Calculate traditional citations (Raw in-degree)
raw_citations = torch.bincount(target_nodes, minlength=num_nodes).float()

# Calculate Semantic Scores (Sum of incoming semantic weights)
semantic_scores = torch.bincount(target_nodes, weights=edge_weights, minlength=num_nodes)

print("5. Generating Final Ranking Comparison...")
# We will use the original parquet file just to get the actual Paper IDs back
df_original = pd.read_parquet("scicite_training_data.parquet", engine="fastparquet")
unique_papers = pd.concat([df_original['citingPaperId'], df_original['citedPaperId']]).unique()

results_df = pd.DataFrame({
    'Paper_ID': unique_papers,
    'Raw_Citations': raw_citations.cpu().numpy(),
    'Semantic_Score': semantic_scores.cpu().numpy()
})

# Filter out papers with 0 citations in this specific subgraph
results_df = results_df[results_df['Raw_Citations'] > 0]

# Calculate the "Impact Multiplier" (How much their score changed based on intent)
results_df['Impact_Multiplier'] = results_df['Semantic_Score'] / results_df['Raw_Citations']

# Sort by the new Semantic Score to see who truly leads the field
results_df = results_df.sort_values(by='Semantic_Score', ascending=False).reset_index(drop=True)

print("\n--- Top 10 Papers by Semantic Score ---")
print(results_df.head(10))

# Save the final baseline output
results_df.to_csv("baseline_semantic_rankings.csv", index=False)
print("\nSaved baseline_semantic_rankings.csv successfully.")

1. Loading Model and Graph...
2. Predicting All Edge Intents...
3. Mapping Intents to Semantic Weights...
4. Calculating Semantic hp-frac Scores...
5. Generating Final Ranking Comparison...

--- Top 10 Papers by Semantic Score ---
                                   Paper_ID  Raw_Citations  Semantic_Score  \
0                                      None          106.0      126.799881   
1  fe13f843c52243e096d7e1a52345a68eaf2cfef1           17.0       19.000000   
2  8d588720ea1b676f8b3906b696e06cf64f5b766b           16.0       17.700001   
3  21096eb950bdff77308ad3924d69b681589bdc91           17.0       17.700001   
4  60ed4bdabf92b2fbd6162dbd8979888cccca55d7           15.0       17.000000   
5  d55c95cedf9904a2776b682518c24b4d4d96bec7           17.0       17.000000   
6  efe167e81022b9fb2b41ea7de996515c26f94838           14.0       16.700001   
7  b0fe8d2727db77f58bc430fef29aba3308ecd3f2           16.0       16.200001   
8  61ba484c3fd142d5ddf606eae1a549fe29594208           15.0       15

### GAT

In [23]:
import torch
import pandas as pd

print("1. Loading the original Parquet data and PyTorch Graph...")
df = pd.read_parquet("scicite_training_data.parquet", engine="fastparquet")

# Remember our weights_only=False fix for PyTorch 2.6!
graph_data = torch.load('scicite_graph.pt', weights_only=False)

print("2. Extracting 'isKeyCitation' labels...")
# The dataset has a boolean column. We convert True/False into 1.0 / 0.0 floats
# We use .view(-1, 1) to make it a 2D tensor of shape [8243, 1] for the neural network head
is_key_array = df['isKeyCitation'].fillna(False).astype(float).values
y_importance = torch.tensor(is_key_array, dtype=torch.float).view(-1, 1)

print(f"Importance Tensor Shape: {y_importance.shape} (Should match Edge count)")

print("3. Attaching to Graph and Saving...")
# Attach it as a brand new attribute
graph_data.y_importance = y_importance

# Let's save it as a new file so we don't overwrite the Phase 1 baseline
torch.save(graph_data, 'scicite_graph_dual.pt')

print("\n--- The Upgraded Dual-Target Graph ---")
print(graph_data)
print("\nGraph saved successfully as 'scicite_graph_dual.pt'")

1. Loading the original Parquet data and PyTorch Graph...
2. Extracting 'isKeyCitation' labels...
Importance Tensor Shape: torch.Size([8243, 1]) (Should match Edge count)
3. Attaching to Graph and Saving...

--- The Upgraded Dual-Target Graph ---
Data(x=[9221, 1], edge_index=[2, 8243], edge_attr=[8243, 768], y=[8243], y_importance=[8243, 1])

Graph saved successfully as 'scicite_graph_dual.pt'


In [26]:
import torch
import numpy as np
import torch.nn as nn
from model_gat import SemanticDualHeadGAT

print("1. Loading Dual-Target Graph and Model...")
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
graph_data = torch.load('scicite_graph_dual.pt', map_location=device, weights_only=False)

model = SemanticDualHeadGAT(
    node_in_dim=1, 
    edge_in_dim=768, 
    hidden_dim=64, 
    num_intents=3
).to(device)

print("2. Generating Edge Splits...")
num_edges = graph_data.num_edges
np.random.seed(42) # For reproducible splits
indices = np.random.permutation(num_edges)

train_idx = indices[:int(0.8 * num_edges)]
val_idx = indices[int(0.8 * num_edges):int(0.9 * num_edges)]
test_idx = indices[int(0.9 * num_edges):]

graph_data.train_mask = torch.zeros(num_edges, dtype=torch.bool).to(device)
graph_data.val_mask = torch.zeros(num_edges, dtype=torch.bool).to(device)
graph_data.test_mask = torch.zeros(num_edges, dtype=torch.bool).to(device)

graph_data.train_mask[train_idx] = True
graph_data.val_mask[val_idx] = True
graph_data.test_mask[test_idx] = True

print("3. Setting up Balanced Dual-Loss Objectives...")
# --- Handle the Class Imbalance ---
class_counts = torch.bincount(graph_data.y)
class_weights = 1.0 / class_counts.float()
# Normalize weights so they sum to the number of classes (3)
class_weights = class_weights / class_weights.sum() * len(class_counts)
print(f"Calculated Class Weights: {class_weights.cpu().numpy()}")

# Head 1 Loss: Weighted Cross Entropy for the Intents
criterion_intent = nn.CrossEntropyLoss(weight=class_weights.to(device))

# Head 2 Loss: Binary Cross Entropy for the Continuous Importance Score
criterion_importance = nn.BCELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

def train():
    model.train()
    optimizer.zero_grad()
    
    intent_logits, importance_score = model(graph_data.x.to(device), graph_data.edge_index.to(device), graph_data.edge_attr.to(device))
    
    # Calculate losses only on the training edges
    loss_intent = criterion_intent(intent_logits[graph_data.train_mask], graph_data.y[graph_data.train_mask].to(device))
    loss_importance = criterion_importance(importance_score[graph_data.train_mask], graph_data.y_importance[graph_data.train_mask].to(device))
    
    # The Multi-Task Objective
    # We weight the importance loss slightly lower so it doesn't overpower the categorical classification
    total_loss = loss_intent + (0.5 * loss_importance)
    
    total_loss.backward()
    optimizer.step()
    return total_loss.item(), loss_intent.item(), loss_importance.item()

@torch.no_grad()
def test(mask):
    model.eval()
    intent_logits, importance_score = model(graph_data.x.to(device), graph_data.edge_index.to(device), graph_data.edge_attr.to(device))
    
    # Intent Accuracy
    pred_intents = intent_logits.argmax(dim=1)
    correct = pred_intents[mask] == graph_data.y[mask].to(device)
    acc = int(correct.sum()) / int(mask.sum())
    
    # Importance Mean Absolute Error (MAE) - How close is the predicted float to the true float?
    true_importance = graph_data.y_importance[mask].to(device)
    mae = torch.abs(importance_score[mask] - true_importance).mean().item()
    
    return acc, mae

print("\n4. Starting Multi-Task Training Loop...")
epochs = 100

for epoch in range(1, epochs + 1):
    t_loss, l_int, l_imp = train()
    
    if epoch % 10 == 0:
        val_acc, val_mae = test(graph_data.val_mask)
        print(f'Epoch: {epoch:03d} | Total Loss: {t_loss:.4f} | Val Intent Acc: {val_acc:.4f} | Val Importance MAE: {val_mae:.4f}')

print("\n5. Final Evaluation...")
test_acc, test_mae = test(graph_data.test_mask)
print(f'Final Test Intent Accuracy: {test_acc:.4f}')
print(f'Final Test Importance Error (MAE): {test_mae:.4f}')

torch.save(model.state_dict(), 'semantic_dual_head_gat.pth')
print("Dual-Head GAT weights saved successfully.")

1. Loading Dual-Target Graph and Model...
2. Generating Edge Splits...
3. Setting up Balanced Dual-Loss Objectives...
Calculated Class Weights: [0.40138376 0.8468603  1.7517561 ]

4. Starting Multi-Task Training Loop...
Epoch: 010 | Total Loss: 0.9437 | Val Intent Acc: 0.7828 | Val Importance MAE: 0.4693
Epoch: 020 | Total Loss: 0.8851 | Val Intent Acc: 0.7779 | Val Importance MAE: 0.4643
Epoch: 030 | Total Loss: 0.8528 | Val Intent Acc: 0.7949 | Val Importance MAE: 0.4632
Epoch: 040 | Total Loss: 0.8311 | Val Intent Acc: 0.7900 | Val Importance MAE: 0.4641
Epoch: 050 | Total Loss: 0.8217 | Val Intent Acc: 0.7961 | Val Importance MAE: 0.4638
Epoch: 060 | Total Loss: 0.8199 | Val Intent Acc: 0.7985 | Val Importance MAE: 0.4626
Epoch: 070 | Total Loss: 0.8179 | Val Intent Acc: 0.8010 | Val Importance MAE: 0.4608
Epoch: 080 | Total Loss: 0.8087 | Val Intent Acc: 0.8046 | Val Importance MAE: 0.4611
Epoch: 090 | Total Loss: 0.8078 | Val Intent Acc: 0.8058 | Val Importance MAE: 0.4606
Epoch:

In [2]:
import pandas as pd
import torch
import requests
import time
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm # Install with: pip install tqdm

print("1. Loading Existing Graph...")
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
graph_data = torch.load('scicite_graph_dual.pt', weights_only=False)

# Get the list of all unique paper IDs from the original Parquet file
df_orig = pd.read_parquet("scicite_training_data.parquet", engine="fastparquet")
unique_papers = pd.concat([df_orig['citingPaperId'], df_orig['citedPaperId']]).unique()

print(f"2. Fetching Abstracts from OpenAlex for {len(unique_papers)} nodes...")
# We will use OpenAlex's fast API. We need a dictionary to store the retrieved abstracts.
abstract_dict = {}

# OpenAlex requires an email in the header to enter the "polite pool" (faster, no 429 limits)
headers = {"User-Agent": "mailto:sanjana.hukkeri@research.iiit.ac.in"} 

# In a production run, you'd batch this, but for this step we will pull a sample to demonstrate
# We will fetch the first 100 to prove the pipeline, then you can run the full batch.
test_papers = unique_papers[:100]

for paper_id in tqdm(test_papers, desc="Fetching Metadata"):
    # We query OpenAlex using the Semantic Scholar ID (mag/s2 formatting)
    url = f"https://api.openalex.org/works?filter=ids.mag:{paper_id}"
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            data = response.json()
            if data['results'] and data['results'][0].get('abstract_inverted_index'):
                # OpenAlex stores abstracts as inverted indices to save space; we reconstruct it
                inv_idx = data['results'][0]['abstract_inverted_index']
                words = max([max(positions) for positions in inv_idx.values()]) + 1
                abstract_text = [""] * words
                for word, positions in inv_idx.items():
                    for pos in positions:
                        abstract_text[pos] = word
                abstract_dict[paper_id] = " ".join(abstract_text)
            else:
                abstract_dict[paper_id] = "No abstract available."
        time.sleep(0.1) # Polite pause
    except Exception as e:
        abstract_dict[paper_id] = "No abstract available."

print("\n3. Generating Node Embeddings via SciBERT...")
tokenizer = AutoTokenizer.from_pretrained('allenai/scibert_scivocab_uncased')
model = AutoModel.from_pretrained('allenai/scibert_scivocab_uncased').to(device)
model.eval()

# To keep math happy, if we don't have the abstract, we feed the model a blank string
# so it at least generates a neutral 768-dim vector instead of a crash.
node_texts = [abstract_dict.get(pid, "Scientific research paper.") for pid in unique_papers]

all_node_embeddings = []
batch_size = 32

with torch.no_grad():
    for i in tqdm(range(0, len(node_texts), batch_size), desc="Encoding Abstracts"):
        batch = node_texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=256, return_tensors="pt").to(device)
        outputs = model(**inputs)
        # Use the [CLS] token for the whole abstract representation
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        all_node_embeddings.append(cls_embeddings.cpu())

# Combine into the final feature matrix X
new_x = torch.cat(all_node_embeddings, dim=0)
print(f"\nNew Node Features Shape: {new_x.shape} (Should be [{len(unique_papers)}, 768])")

print("4. Attaching and Saving the Final Graph...")
graph_data.x = new_x
torch.save(graph_data, 'scicite_graph_final.pt')
print("Graph saved successfully as 'scicite_graph_final.pt'.")

/Users/sanjanahukkeri/Documents/DS/DS Project Repo/DS-ResearchPaperPipeline/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. Loading Existing Graph...
2. Fetching Abstracts from OpenAlex for 9221 nodes...


Fetching Metadata: 100%|██████████| 100/100 [01:12<00:00,  1.37it/s]



3. Generating Node Embeddings via SciBERT...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 68662.92it/s]
BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Encoding Abstracts: 100%|██████████| 289/289 [00:03<00:00, 77.78it


New Node Features Shape: torch.Size([9221, 768]) (Should be [9221, 768])
4. Attaching and Saving the Final Graph...
Graph saved successfully as 'scicite_graph_final.pt'.
